<a href="https://colab.research.google.com/github/hasmalee/aeropinn/blob/main/RMSE_for_L_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import math
import numpy as np
import pandas as pd
from openpyxl import load_workbook

In [2]:
BOOK_PATH = "/mnt/data/Book1.xlsx"
CSV_PATH  = "/mnt/data/airfoil_aerodynamic_coefficients_100.csv"   # optional cross-check

In [3]:
import re
import math
import numpy as np
import pandas as pd
from openpyxl import load_workbook

BOOK_PATH = "/content/Book1.xlsx"   # change if needed

wb = load_workbook(BOOK_PATH, data_only=True)
ws = wb[wb.sheetnames[0]]
rows = list(ws.iter_rows(values_only=True))

print("Rows:", len(rows))
for i, r in enumerate(rows[:15], start=1):
    print(i, r)

Rows: 197
1 (None, None, None, None, None)
2 (' naca0006_a4_Re2000000', 'PINN Value', 'LBFGS value', 'CFD Value', 'https://www.icas.org/icas_archive/ICAS2014/data/papers/2014_0818_paper.pdf')
3 ('CL', 0.3938, 0.2453, 0.44, None)
4 ('CD', 0.00768, 0.01873, 0.0085, None)
5 ('L/D', 51.261, 13.099, 51.76, None)
6 (None, None, None, None, None)
7 (None, None, None, None, None)
8 ('naca0006_a0_Re2000000', 'PINN Value', 'LBFGS value', 'CFD Value', 'http://airfoiltools.com/polar/details?polar=xf-naca0006-il-1000000')
9 ('CL', 0, 0, 0, None)
10 ('CD', 0, 0.008, 0.0062, None)
11 ('L/D', -3.157, -0.001, 0, None)
12 (None, None, None, None, None)
13 (None, None, None, None, None)
14 ('naca0009_a0_Re1000000', 'PINN Value', 'LBFGS value', 'CFD Value', 'https://www.google.com/search?q=http://airfoiltools.com/airfoil/details%3Fairfoil%3Dnaca0009-il')
15 ('CL', -0.0164, -0.0131, 0.525, None)


In [4]:
def parse_case_name(case_name: str):
    case_name = str(case_name).strip()
    m = re.match(r"(.+)_a(-?\d+(?:\.\d+)?)_Re(\d+)", case_name)
    if not m:
        return None, None, None
    return m.group(1).strip(), float(m.group(2)), float(m.group(3))

records = []
i = 0

while i < len(rows):
    first = rows[i][0]

    if isinstance(first, str) and "_a" in first and "_Re" in first:
        airfoil, aoa, Re = parse_case_name(first)

        rec = {
            "case_name": str(first).strip(),
            "airfoil": airfoil,
            "aoa": aoa,
            "Re": Re,
        }

        # next rows: CL, CD, L/D
        for j in range(i + 1, min(i + 5, len(rows))):
            metric = rows[j][0]
            if metric is None:
                continue

            metric = str(metric).strip().upper().replace(" ", "")
            pinn = rows[j][1]
            opt  = rows[j][2]
            cfd  = rows[j][3]

            if metric == "CL":
                rec["CL_original"] = pinn
                rec["CL_optimized"] = opt
                rec["CL_cfd"] = cfd

            elif metric == "CD":
                rec["CD_original"] = pinn
                rec["CD_optimized"] = opt
                rec["CD_cfd"] = cfd

            elif metric in ["L/D", "LD"]:
                rec["LD_original"] = pinn
                rec["LD_optimized"] = opt
                rec["LD_cfd"] = cfd

        records.append(rec)
        i += 4
    else:
        i += 1

df = pd.DataFrame(records)
print("Cases found:", len(df))
df.head(10)

Cases found: 20


,case_name,airfoil,aoa,Re,CL_original,CL_optimized,CL_cfd,CD_original,CD_optimized,CD_cfd,LD_original,LD_optimized,LD_cfd
0,naca0006_a4_Re2000000,naca0006,4.0,2000000.0,0.3938,0.2453,0.440,0.00768,0.01873,0.0085,51.261,13.099,51.7600
1,naca0006_a0_Re2000000,naca0006,0.0,2000000.0,0.0000,0.0000,0.000,0.00000,0.00800,0.0062,-3.157,-0.001,0.0000
2,naca0009_a0_Re1000000,naca0009,0.0,1000000.0,-0.0164,-0.0131,0.525,0.00060,0.00800,0.0078,-265.639,-1.633,0.0000
3,naca0009_a3_Re1000000,naca0009,3.0,1000000.0,0.0391,0.1061,0.325,0.00446,0.00800,0.0098,8.761,13.265,33.1600
4,naca0009_a4_Re2000000,naca0009,4.0,2000000.0,0.5204,0.3958,0.445,0.01435,0.00800,0.0085,36.266,49.476,52.3500
5,naca0012-64_a4_Re2000000,naca0012-64,4.0,2000000.0,0.5742,0.5135,0.125,0.00948,0.02602,0.0085,60.556,19.735,14.7100
6,naca0015_a4_Re2000000,naca0015,4.0,2000000.0,0.6555,0.6937,0.442,0.02640,0.02450,0.0071,24.828,28.317,62.2500
7,naca0015_a2_Re1000000,naca0015,2.0,1000000.0,0.1535,0.3465,0.218,0.02822,0.00800,0.0084,5.439,43.318,25.9500
8,naca0018_a4_Re2000000,naca0018,4.0,2000000.0,0.4628,0.6922,0.435,0.01189,0.02199,0.0088,38.915,31.483,0.0088
9,naca64_012_a0_Re2000000,naca64_012,0.0,2000000.0,0.0000,0.0000,0.000,0.00000,0.00800,0.0048,-3.258,-0.001,0.0000


In [5]:
def abs_err(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return abs(float(a) - float(b))

def pct_err(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    b = float(b)
    if abs(b) < 1e-12:   # avoid divide by zero
        return np.nan
    return abs(float(a) - b) / abs(b) * 100.0

# -------- CL --------
df["CL_abs_err_orig_cfd"] = df.apply(lambda r: abs_err(r["CL_original"], r["CL_cfd"]), axis=1)
df["CL_abs_err_opt_cfd"]  = df.apply(lambda r: abs_err(r["CL_optimized"], r["CL_cfd"]), axis=1)
df["CL_abs_err_opt_orig"] = df.apply(lambda r: abs_err(r["CL_optimized"], r["CL_original"]), axis=1)

df["CL_pct_err_orig_cfd"] = df.apply(lambda r: pct_err(r["CL_original"], r["CL_cfd"]), axis=1)
df["CL_pct_err_opt_cfd"]  = df.apply(lambda r: pct_err(r["CL_optimized"], r["CL_cfd"]), axis=1)
df["CL_pct_err_opt_orig"] = df.apply(lambda r: pct_err(r["CL_optimized"], r["CL_original"]), axis=1)

# -------- CD --------
df["CD_abs_err_orig_cfd"] = df.apply(lambda r: abs_err(r["CD_original"], r["CD_cfd"]), axis=1)
df["CD_abs_err_opt_cfd"]  = df.apply(lambda r: abs_err(r["CD_optimized"], r["CD_cfd"]), axis=1)
df["CD_abs_err_opt_orig"] = df.apply(lambda r: abs_err(r["CD_optimized"], r["CD_original"]), axis=1)

df["CD_pct_err_orig_cfd"] = df.apply(lambda r: pct_err(r["CD_original"], r["CD_cfd"]), axis=1)
df["CD_pct_err_opt_cfd"]  = df.apply(lambda r: pct_err(r["CD_optimized"], r["CD_cfd"]), axis=1)
df["CD_pct_err_opt_orig"] = df.apply(lambda r: pct_err(r["CD_optimized"], r["CD_original"]), axis=1)

# -------- L/D --------
df["LD_abs_err_orig_cfd"] = df.apply(lambda r: abs_err(r["LD_original"], r["LD_cfd"]), axis=1)
df["LD_abs_err_opt_cfd"]  = df.apply(lambda r: abs_err(r["LD_optimized"], r["LD_cfd"]), axis=1)
df["LD_abs_err_opt_orig"] = df.apply(lambda r: abs_err(r["LD_optimized"], r["LD_original"]), axis=1)

df["LD_pct_err_orig_cfd"] = df.apply(lambda r: pct_err(r["LD_original"], r["LD_cfd"]), axis=1)
df["LD_pct_err_opt_cfd"]  = df.apply(lambda r: pct_err(r["LD_optimized"], r["LD_cfd"]), axis=1)
df["LD_pct_err_opt_orig"] = df.apply(lambda r: pct_err(r["LD_optimized"], r["LD_original"]), axis=1)

# best comparison labels
def best_of_three(a, b, c, labels=("Original vs CFD", "Optimized vs CFD", "Optimized vs Original")):
    vals = [a, b, c]
    valid = [(v, labels[i]) for i, v in enumerate(vals) if not pd.isna(v)]
    if not valid:
        return np.nan
    return min(valid, key=lambda x: x[0])[1]

df["CL_best_match"] = df.apply(lambda r: best_of_three(
    r["CL_pct_err_orig_cfd"], r["CL_pct_err_opt_cfd"], r["CL_pct_err_opt_orig"]), axis=1)

df["CD_best_match"] = df.apply(lambda r: best_of_three(
    r["CD_pct_err_orig_cfd"], r["CD_pct_err_opt_cfd"], r["CD_pct_err_opt_orig"]), axis=1)

df["LD_best_match"] = df.apply(lambda r: best_of_three(
    r["LD_pct_err_orig_cfd"], r["LD_pct_err_opt_cfd"], r["LD_pct_err_opt_orig"]), axis=1)

per_airfoil_table = df[[
    "case_name", "airfoil", "aoa", "Re",

    "CL_original", "CL_optimized", "CL_cfd",
    "CD_original", "CD_optimized", "CD_cfd",
    "LD_original", "LD_optimized", "LD_cfd",

    "CL_abs_err_orig_cfd", "CL_abs_err_opt_cfd", "CL_abs_err_opt_orig",
    "CL_pct_err_orig_cfd", "CL_pct_err_opt_cfd", "CL_pct_err_opt_orig", "CL_best_match",

    "CD_abs_err_orig_cfd", "CD_abs_err_opt_cfd", "CD_abs_err_opt_orig",
    "CD_pct_err_orig_cfd", "CD_pct_err_opt_cfd", "CD_pct_err_opt_orig", "CD_best_match",

    "LD_abs_err_orig_cfd", "LD_abs_err_opt_cfd", "LD_abs_err_opt_orig",
    "LD_pct_err_orig_cfd", "LD_pct_err_opt_cfd", "LD_pct_err_opt_orig", "LD_best_match",
]].copy()

per_airfoil_table.head(10)

,case_name,airfoil,aoa,Re,CL_original,CL_optimized,CL_cfd,CD_original,CD_optimized,CD_cfd,...,CD_pct_err_opt_cfd,CD_pct_err_opt_orig,CD_best_match,LD_abs_err_orig_cfd,LD_abs_err_opt_cfd,LD_abs_err_opt_orig,LD_pct_err_orig_cfd,LD_pct_err_opt_cfd,LD_pct_err_opt_orig,LD_best_match
0,naca0006_a4_Re2000000,naca0006,4.0,2000000.0,0.3938,0.2453,0.440,0.00768,0.01873,0.0085,...,120.352941,143.880208,Original vs CFD,0.4990,38.6610,38.162,0.964065,74.692813,74.446460,Original vs CFD
1,naca0006_a0_Re2000000,naca0006,0.0,2000000.0,0.0000,0.0000,0.000,0.00000,0.00800,0.0062,...,29.032258,NaN,Optimized vs CFD,3.1570,0.0010,3.156,NaN,NaN,99.968324,Optimized vs Original
2,naca0009_a0_Re1000000,naca0009,0.0,1000000.0,-0.0164,-0.0131,0.525,0.00060,0.00800,0.0078,...,2.564103,1233.333333,Optimized vs CFD,265.6390,1.6330,264.006,NaN,NaN,99.385256,Optimized vs Original
3,naca0009_a3_Re1000000,naca0009,3.0,1000000.0,0.0391,0.1061,0.325,0.00446,0.00800,0.0098,...,18.367347,79.372197,Optimized vs CFD,24.3990,19.8950,4.504,73.579614,59.996984,51.409656,Optimized vs Original
4,naca0009_a4_Re2000000,naca0009,4.0,2000000.0,0.5204,0.3958,0.445,0.01435,0.00800,0.0085,...,5.882353,44.250871,Optimized vs CFD,16.0840,2.8740,13.210,30.723973,5.489971,36.425302,Optimized vs CFD
5,naca0012-64_a4_Re2000000,naca0012-64,4.0,2000000.0,0.5742,0.5135,0.125,0.00948,0.02602,0.0085,...,206.117647,174.472574,Original vs CFD,45.8460,5.0250,40.821,311.665534,34.160435,67.410331,Optimized vs CFD
6,naca0015_a4_Re2000000,naca0015,4.0,2000000.0,0.6555,0.6937,0.442,0.02640,0.02450,0.0071,...,245.070423,7.196970,Optimized vs Original,37.4220,33.9330,3.489,60.115663,54.510843,14.052682,Optimized vs Original
7,naca0015_a2_Re1000000,naca0015,2.0,1000000.0,0.1535,0.3465,0.218,0.02822,0.00800,0.0084,...,4.761905,71.651311,Optimized vs CFD,20.5110,17.3680,37.879,79.040462,66.928709,696.433168,Optimized vs CFD
8,naca0018_a4_Re2000000,naca0018,4.0,2000000.0,0.4628,0.6922,0.435,0.01189,0.02199,0.0088,...,149.886364,84.945332,Original vs CFD,38.9062,31.4742,7.432,442115.909091,357661.363636,19.098034,Optimized vs Original
9,naca64_012_a0_Re2000000,naca64_012,0.0,2000000.0,0.0000,0.0000,0.000,0.00000,0.00800,0.0048,...,66.666667,NaN,Optimized vs CFD,3.2580,0.0010,3.257,NaN,NaN,99.969306,Optimized vs Original


In [6]:
def rmse_from_cols(df, pred_col, true_col):
    tmp = df[[pred_col, true_col]].dropna().copy()
    if len(tmp) == 0:
        return np.nan
    return math.sqrt(np.mean((tmp[pred_col].astype(float) - tmp[true_col].astype(float)) ** 2))

rmse_summary = pd.DataFrame([
    {
        "Metric": "CL",
        "RMSE_Original_vs_CFD": rmse_from_cols(df, "CL_original", "CL_cfd"),
        "RMSE_Optimized_vs_CFD": rmse_from_cols(df, "CL_optimized", "CL_cfd"),
        "RMSE_Optimized_vs_Original": rmse_from_cols(df, "CL_optimized", "CL_original"),
    },
    {
        "Metric": "CD",
        "RMSE_Original_vs_CFD": rmse_from_cols(df, "CD_original", "CD_cfd"),
        "RMSE_Optimized_vs_CFD": rmse_from_cols(df, "CD_optimized", "CD_cfd"),
        "RMSE_Optimized_vs_Original": rmse_from_cols(df, "CD_optimized", "CD_original"),
    },
    {
        "Metric": "L/D",
        "RMSE_Original_vs_CFD": rmse_from_cols(df, "LD_original", "LD_cfd"),
        "RMSE_Optimized_vs_CFD": rmse_from_cols(df, "LD_optimized", "LD_cfd"),
        "RMSE_Optimized_vs_Original": rmse_from_cols(df, "LD_optimized", "LD_original"),
    }
])

rmse_summary

,Metric,RMSE_Original_vs_CFD,RMSE_Optimized_vs_CFD,RMSE_Optimized_vs_Original
0,CL,0.357609,0.277416,0.173588
1,CD,0.023796,0.017012,0.017396
2,L/D,95.033328,32.925985,88.031255


separate tables — only **L/D optimized vs CFD**

In [9]:

import pandas as pd
import re
from IPython.display import display, Markdown

with open('/content/Pasted text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Split into blocks starting at each case name
blocks = re.split(r'\n(?=naca)', text)

ld_tables = {}

for block in blocks:
    header_match = re.search(r'(naca[\w\-]+)_a([\d\.]+)_Re(\d+)', block)
    if not header_match:
        continue

    airfoil = header_match.group(1)
    aoa = float(header_match.group(2))
    re_val = int(header_match.group(3))
    case_name = f"{airfoil}_a{aoa:g}_Re{re_val}"

    # Extract only the L/D row from the pasted tables
    ld_match = re.search(r'\bL/D\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)', block)
    if not ld_match:
        continue

    ld_opt_vs_cfd = float(ld_match.group(2))

    title = f"{airfoil} | AoA={aoa} | Re={re_val}"
    ld_tables[title] = pd.DataFrame({
        "Metric": ["L/D"],
        "RMSE (Optimized vs CFD)": [ld_opt_vs_cfd]
    })

print(f"Total airfoil cases found: {len(ld_tables)}")


Total airfoil cases found: 20


In [10]:

for name, table in ld_tables.items():
    print("=" * 60)
    print(name)
    display(table.round(6))


naca0006 | AoA=4.0 | Re=2000000


,Metric,RMSE (Optimized vs CFD)
0,L/D,38.661


naca0006 | AoA=0.0 | Re=2000000


,Metric,RMSE (Optimized vs CFD)
0,L/D,0.001


naca0009 | AoA=0.0 | Re=1000000


...

,Metric,RMSE (Optimized vs CFD)
0,L/D,50.2


naca4412 | AoA=4.0 | Re=1000000


,Metric,RMSE (Optimized vs CFD)
0,L/D,44.046


naca23015 | AoA=4.0 | Re=1000000


,Metric,RMSE (Optimized vs CFD)
0,L/D,55.634


In [11]:

summary_rows = []

for name, table in ld_tables.items():
    summary_rows.append({
        "Case": name,
        "Metric": "L/D",
        "RMSE (Optimized vs CFD)": float(table.loc[0, "RMSE (Optimized vs CFD)"])
    })

summary_df = pd.DataFrame(summary_rows)

print("L/D optimized-vs-CFD summary:")
display(summary_df.round(6))


L/D optimized-vs-CFD summary:


,Case,Metric,RMSE (Optimized vs CFD)
0,naca0006 | AoA=4.0 | Re=2000000,L/D,38.6610
1,naca0006 | AoA=0.0 | Re=2000000,L/D,0.0010
2,naca0009 | AoA=0.0 | Re=1000000,L/D,1.6330
3,naca0009 | AoA=3.0 | Re=1000000,L/D,19.8950
4,naca0009 | AoA=4.0 | Re=2000000,L/D,2.8740
5,naca0012-64 | AoA=4.0 | Re=2000000,L/D,5.0250
6,naca0015 | AoA=4.0 | Re=2000000,L/D,33.9330
7,naca0015 | AoA=2.0 | Re=1000000,L/D,17.3680
8,naca0018 | AoA=4.0 | Re=2000000,L/D,31.4742
9,naca64_012 | AoA=0.0 | Re=2000000,L/D,0.0010


In [12]:

avg_rmse = summary_df["RMSE (Optimized vs CFD)"].mean()
median_rmse = summary_df["RMSE (Optimized vs CFD)"].median()
min_rmse = summary_df["RMSE (Optimized vs CFD)"].min()
max_rmse = summary_df["RMSE (Optimized vs CFD)"].max()

stats_df = pd.DataFrame([{
    "Metric": "L/D",
    "Mean RMSE (Optimized vs CFD)": avg_rmse,
    "Median RMSE (Optimized vs CFD)": median_rmse,
    "Min RMSE (Optimized vs CFD)": min_rmse,
    "Max RMSE (Optimized vs CFD)": max_rmse
}])

print("Overall L/D RMSE statistics:")
display(stats_df.round(6))


Overall L/D RMSE statistics:


,Metric,Mean RMSE (Optimized vs CFD),Median RMSE (Optimized vs CFD),Min RMSE (Optimized vs CFD),Max RMSE (Optimized vs CFD)
0,L/D,27.50486,30.575,0.001,55.634


In [13]:

# Change threshold as needed
threshold = 50

bad_ld = summary_df[summary_df["RMSE (Optimized vs CFD)"] > threshold].copy()

print(f"Cases with L/D optimized-vs-CFD RMSE > {threshold}:")
display(bad_ld.round(6))


Cases with L/D optimized-vs-CFD RMSE > 50:


,Case,Metric,RMSE (Optimized vs CFD)
13,naca2415 | AoA=4.0 | Re=2000000,L/D,51.359
17,naca0018 | AoA=4.0 | Re=1000000,L/D,50.200
19,naca23015 | AoA=4.0 | Re=1000000,L/D,55.634


In [14]:

OUT_CSV = "/content/ld_optimized_vs_cfd_tables.csv"
OUT_XLSX = "/content/ld_optimized_vs_cfd_tables.xlsx"

summary_df.to_csv(OUT_CSV, index=False)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    for name, table in ld_tables.items():
        sheet_name = name[:31]
        table.round(6).to_excel(writer, sheet_name=sheet_name, index=False)
    summary_df.round(6).to_excel(writer, sheet_name="summary", index=False)

print("Saved files:")
print(OUT_CSV)
print(OUT_XLSX)


Saved files:
/content/ld_optimized_vs_cfd_tables.csv
/content/ld_optimized_vs_cfd_tables.xlsx


Normalized RMSE

In [24]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook

BOOK_PATH = "/content/Book1.xlsx"   # change if needed

wb = load_workbook(BOOK_PATH, data_only=True)
ws = wb[wb.sheetnames[0]]
rows = list(ws.iter_rows(values_only=True))

print("Rows found:", len(rows))
for i, r in enumerate(rows[:15], start=1):
    print(i, r)

Rows found: 197
1 (None, None, None, None, None)
2 (' naca0006_a4_Re2000000', 'PINN Value', 'LBFGS value', 'CFD Value', 'https://www.icas.org/icas_archive/ICAS2014/data/papers/2014_0818_paper.pdf')
3 ('CL', 0.3938, 0.2453, 0.44, None)
4 ('CD', 0.00768, 0.01873, 0.0085, None)
5 ('L/D', 51.261, 13.099, 51.76, None)
6 (None, None, None, None, None)
7 (None, None, None, None, None)
8 ('naca0006_a0_Re2000000', 'PINN Value', 'LBFGS value', 'CFD Value', 'http://airfoiltools.com/polar/details?polar=xf-naca0006-il-1000000')
9 ('CL', 0, 0, 0, None)
10 ('CD', 0, 0.008, 0.0062, None)
11 ('L/D', -3.157, -0.001, 0, None)
12 (None, None, None, None, None)
13 (None, None, None, None, None)
14 ('naca0009_a0_Re1000000', 'PINN Value', 'LBFGS value', 'CFD Value', 'https://www.google.com/search?q=http://airfoiltools.com/airfoil/details%3Fairfoil%3Dnaca0009-il')
15 ('CL', -0.0164, -0.0131, 0.525, None)


In [25]:
def parse_case_name(case_name: str):
    case_name = str(case_name).strip()
    m = re.match(r"(.+)_a(-?\d+(?:\.\d+)?)_Re(\d+)", case_name)
    if not m:
        return None, None, None
    return m.group(1).strip(), float(m.group(2)), float(m.group(3))

records = []
i = 0

while i < len(rows):
    first = rows[i][0]

    if isinstance(first, str) and "_a" in first and "_Re" in first:
        airfoil, aoa, Re = parse_case_name(first)

        rec = {
            "case_name": str(first).strip(),
            "airfoil": airfoil,
            "aoa": aoa,
            "Re": Re,
        }

        for j in range(i + 1, min(i + 6, len(rows))):
            metric = rows[j][0]
            if metric is None:
                continue

            metric_clean = str(metric).strip().upper().replace(" ", "")
            opt = rows[j][2]
            cfd = rows[j][3]

            if metric_clean == "CL":
                rec["CL_optimized"] = float(opt)
                rec["CL_cfd"] = float(cfd)

            elif metric_clean == "CD":
                rec["CD_optimized"] = float(opt)
                rec["CD_cfd"] = float(cfd)

            elif metric_clean in ["L/D", "LD"]:
                rec["LD_optimized"] = float(opt)
                rec["LD_cfd"] = float(cfd)

        records.append(rec)
        i += 4
    else:
        i += 1

df = pd.DataFrame(records)
print("Cases found:", len(df))
df.head()

Cases found: 20


,case_name,airfoil,aoa,Re,CL_optimized,CL_cfd,CD_optimized,CD_cfd,LD_optimized,LD_cfd
0,naca0006_a4_Re2000000,naca0006,4.0,2000000.0,0.2453,0.440,0.01873,0.0085,13.099,51.76
1,naca0006_a0_Re2000000,naca0006,0.0,2000000.0,0.0000,0.000,0.00800,0.0062,-0.001,0.00
2,naca0009_a0_Re1000000,naca0009,0.0,1000000.0,-0.0131,0.525,0.00800,0.0078,-1.633,0.00
3,naca0009_a3_Re1000000,naca0009,3.0,1000000.0,0.1061,0.325,0.00800,0.0098,13.265,33.16
4,naca0009_a4_Re2000000,naca0009,4.0,2000000.0,0.3958,0.445,0.00800,0.0085,49.476,52.35


In [26]:
EPS = 1e-12

def rmse_single(pred, true):
    return abs(float(pred) - float(true))

def nrmse_single(pred, true):
    true = float(true)
    return abs(float(pred) - true) / max(abs(true), EPS)

for metric in ["CL", "CD", "LD"]:
    df[f"{metric}_RMSE_Opt_vs_CFD"] = df.apply(
        lambda r: rmse_single(r[f"{metric}_optimized"], r[f"{metric}_cfd"]), axis=1
    )
    df[f"{metric}_NRMSE_Opt_vs_CFD"] = df.apply(
        lambda r: nrmse_single(r[f"{metric}_optimized"], r[f"{metric}_cfd"]), axis=1
    )

df.head()

,case_name,airfoil,aoa,Re,CL_optimized,CL_cfd,CD_optimized,CD_cfd,LD_optimized,LD_cfd,CL_RMSE_Opt_vs_CFD,CL_NRMSE_Opt_vs_CFD,CD_RMSE_Opt_vs_CFD,CD_NRMSE_Opt_vs_CFD,LD_RMSE_Opt_vs_CFD,LD_NRMSE_Opt_vs_CFD
0,naca0006_a4_Re2000000,naca0006,4.0,2000000.0,0.2453,0.440,0.01873,0.0085,13.099,51.76,0.1947,0.442500,0.01023,1.203529,38.661,7.469281e-01
1,naca0006_a0_Re2000000,naca0006,0.0,2000000.0,0.0000,0.000,0.00800,0.0062,-0.001,0.00,0.0000,0.000000,0.00180,0.290323,0.001,1.000000e+09
2,naca0009_a0_Re1000000,naca0009,0.0,1000000.0,-0.0131,0.525,0.00800,0.0078,-1.633,0.00,0.5381,1.024952,0.00020,0.025641,1.633,1.633000e+12
3,naca0009_a3_Re1000000,naca0009,3.0,1000000.0,0.1061,0.325,0.00800,0.0098,13.265,33.16,0.2189,0.673538,0.00180,0.183673,19.895,5.999698e-01
4,naca0009_a4_Re2000000,naca0009,4.0,2000000.0,0.3958,0.445,0.00800,0.0085,49.476,52.35,0.0492,0.110562,0.00050,0.058824,2.874,5.489971e-02


In [27]:
OUT_DIR = "/content/per_airfoil_opt_vs_cfd_plots"
os.makedirs(OUT_DIR, exist_ok=True)

metric_labels = ["CL", "CD", "L/D"]

for _, row in df.iterrows():
    case_name = row["case_name"]
    case_dir = os.path.join(OUT_DIR, case_name)
    os.makedirs(case_dir, exist_ok=True)

    # -----------------------------
    # 1) NRMSE plot
    # -----------------------------
    nrmse_vals = [
        row["CL_NRMSE_Opt_vs_CFD"],
        row["CD_NRMSE_Opt_vs_CFD"],
        row["LD_NRMSE_Opt_vs_CFD"],
    ]

    x = np.arange(len(metric_labels))

    plt.figure(figsize=(7, 4))
    plt.bar(x, nrmse_vals)
    plt.xticks(x, metric_labels)
    plt.ylabel("NRMSE")
    plt.title(f"NRMSE (Optimized vs CFD)\n{case_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(case_dir, "nrmse_opt_vs_cfd.png"), dpi=200)
    plt.close()

    # -----------------------------
    # 2) Error distribution plot
    # -----------------------------
    rmse_vals = [
        row["CL_RMSE_Opt_vs_CFD"],
        row["CD_RMSE_Opt_vs_CFD"],
        row["LD_RMSE_Opt_vs_CFD"],
    ]

    plt.figure(figsize=(7, 4))
    plt.bar(x, rmse_vals)
    plt.xticks(x, metric_labels)
    plt.ylabel("RMSE")
    plt.title(f"Error Distribution (Optimized vs CFD)\n{case_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(case_dir, "error_distribution_opt_vs_cfd.png"), dpi=200)
    plt.close()

    # -----------------------------
    # 3) PINN vs CFD normalized trend
    # CFD baseline = 1
    # -----------------------------
    norm_opt = [
        row["CL_optimized"] / max(abs(row["CL_cfd"]), EPS),
        row["CD_optimized"] / max(abs(row["CD_cfd"]), EPS),
        row["LD_optimized"] / max(abs(row["LD_cfd"]), EPS),
    ]
    norm_cfd = [1.0, 1.0, 1.0]

    plt.figure(figsize=(7, 4))
    plt.plot(metric_labels, norm_opt, marker="o", label="Optimized PINN / CFD")
    plt.plot(metric_labels, norm_cfd, marker="s", linestyle="--", label="CFD Baseline")
    plt.ylabel("Normalized Value")
    plt.title(f"Optimized PINN vs CFD Trend\n{case_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(case_dir, "trend_opt_vs_cfd.png"), dpi=200)
    plt.close()

print("Saved plots to:", OUT_DIR)

Saved plots to: /content/per_airfoil_opt_vs_cfd_plots


In [28]:
summary_df = df[[
    "case_name", "airfoil", "aoa", "Re",

    "CL_RMSE_Opt_vs_CFD", "CL_NRMSE_Opt_vs_CFD",
    "CD_RMSE_Opt_vs_CFD", "CD_NRMSE_Opt_vs_CFD",
    "LD_RMSE_Opt_vs_CFD", "LD_NRMSE_Opt_vs_CFD",
]].copy()

summary_df.head()

,case_name,airfoil,aoa,Re,CL_RMSE_Opt_vs_CFD,CL_NRMSE_Opt_vs_CFD,CD_RMSE_Opt_vs_CFD,CD_NRMSE_Opt_vs_CFD,LD_RMSE_Opt_vs_CFD,LD_NRMSE_Opt_vs_CFD
0,naca0006_a4_Re2000000,naca0006,4.0,2000000.0,0.1947,0.442500,0.01023,1.203529,38.661,7.469281e-01
1,naca0006_a0_Re2000000,naca0006,0.0,2000000.0,0.0000,0.000000,0.00180,0.290323,0.001,1.000000e+09
2,naca0009_a0_Re1000000,naca0009,0.0,1000000.0,0.5381,1.024952,0.00020,0.025641,1.633,1.633000e+12
3,naca0009_a3_Re1000000,naca0009,3.0,1000000.0,0.2189,0.673538,0.00180,0.183673,19.895,5.999698e-01
4,naca0009_a4_Re2000000,naca0009,4.0,2000000.0,0.0492,0.110562,0.00050,0.058824,2.874,5.489971e-02


In [33]:
for metric in ["CL", "CD", "LD"]:
    summary_df[f"{metric}_Safe"] = summary_df[f"{metric}_NRMSE_Opt_vs_CFD"] <= 0.20

summary_df.head()

,case_name,airfoil,aoa,Re,CL_RMSE_Opt_vs_CFD,CL_NRMSE_Opt_vs_CFD,CD_RMSE_Opt_vs_CFD,CD_NRMSE_Opt_vs_CFD,LD_RMSE_Opt_vs_CFD,LD_NRMSE_Opt_vs_CFD,CL_Safe,CD_Safe,LD_Safe
0,naca0006_a4_Re2000000,naca0006,4.0,2000000.0,0.1947,0.442500,0.01023,1.203529,38.661,7.469281e-01,False,False,False
1,naca0006_a0_Re2000000,naca0006,0.0,2000000.0,0.0000,0.000000,0.00180,0.290323,0.001,1.000000e+09,True,False,False
2,naca0009_a0_Re1000000,naca0009,0.0,1000000.0,0.5381,1.024952,0.00020,0.025641,1.633,1.633000e+12,False,True,False
3,naca0009_a3_Re1000000,naca0009,3.0,1000000.0,0.2189,0.673538,0.00180,0.183673,19.895,5.999698e-01,False,True,False
4,naca0009_a4_Re2000000,naca0009,4.0,2000000.0,0.0492,0.110562,0.00050,0.058824,2.874,5.489971e-02,True,True,True


In [34]:
for metric in ["CL", "CD", "LD"]:
    safe_count = summary_df[f"{metric}_Safe"].sum()
    total = len(summary_df)
    print(f"{metric} safe cases (Optimized vs CFD): {safe_count}/{total} = {100*safe_count/total:.1f}%")

CL safe cases (Optimized vs CFD): 6/20 = 30.0%
CD safe cases (Optimized vs CFD): 6/20 = 30.0%
LD safe cases (Optimized vs CFD): 1/20 = 5.0%


In [35]:
summary_df.to_csv("/content/optimized_vs_cfd_nrmse_summary.csv", index=False)
df.to_csv("/content/optimized_vs_cfd_full_metrics.csv", index=False)

print("Saved:")
print("/content/optimized_vs_cfd_nrmse_summary.csv")
print("/content/optimized_vs_cfd_full_metrics.csv")

Saved:
/content/optimized_vs_cfd_nrmse_summary.csv
/content/optimized_vs_cfd_full_metrics.csv
